# DFT Static Diagnostics on Final FP-NEB Image Structures

## 1. Purpose and Protocol Definition

This notebook performs DFT static calculations on the final image structures produced by the full FP-NEB workflow. It does not relax structures, run a foundation potential, or use the DFT-NEB reference image structures.

Final image structures, energies, forces, foundation-potential keys, and convergence states are read directly from the `full_fp_neb` branch of the standardized all-FP results JSON. Converged and non-converged full FP-NEB paths remain separately identifiable throughout (`neb_status.neb_converged`); a non-converged path is never excluded from this diagnostic on that basis alone. Generated DFT results populate only the `dft_static_on_fp_neb` branch. Force errors on the FP-NEB path are computed later, by the analysis notebook, comparing these DFT evaluations against the FP values at the same images.


## 2. Using FPBench Results or Another Results File

| Workflow | Structural input | Results branch |
|---|---|---|
| DFT static diagnostics on the final full FP-NEB image structures | Final full FP-NEB image structures | `dft_static_on_fp_neb` |

**Required input file:** `FP_RESULTS_FILE` (Configuration), following the schema below -- the standardized all-FP results file, loaded directly, no reshaping.

```python
fp_results = {
    "models": {
        fp_key: {
            "full_fp_neb": {
                "pathways": {...},
                "unsuccessful_pathways": {...},
            },
            "fp_static_on_dft_neb": {
                "pathways": {...},
                "unsuccessful_pathways": {...},
            },
            "dft_static_on_fp_neb": {
                "pathways": {...},
                "unsuccessful_image_attempts": {...},
            },
        },
    },
}

# To validate against a not-yet-promoted candidate instead of the
# canonical results/ file:
# FP_RESULTS_FILE = Path("/path/to/my_results.json")
```

**Main workflow:** Select eligible full FP-NEB pathways, generate VASP static calculations, submit, parse and validate, then Merge Into the All-Protocol Results File (below) -- which writes only the `dft_static_on_fp_neb` branch into a new file under `runs/`, never overwriting the input. **Detailed schema:** `results/README.md`.

One function builds every DFT-static image record used by job generation, merge validation, and this documentation.

In [ ]:
import json
import numpy as np

ENDPOINT_ROLES = ("initial", "intermediate", "final")

CALCULATION_STATUS_VALUES = ("completed", "partial", "failed", "interrupted", "missing", "not_run")


class MergeError(Exception):
    pass


def endpoint_role_for(image_index, n_images):
    if image_index == 0:
        return "initial"
    if image_index == n_images - 1:
        return "final"
    return "intermediate"


def make_identifiers(source_identifiers):
    return dict(source_identifiers)


def validate_structure_atoms(structure_dict, reference_structure_dict, context,
                              image_index=None, reference_image_index=None,
                              source_identifiers=None, reference_identifiers=None,
                              frac_coord_atol=1e-6, lattice_atol=1e-6):
    # Defined locally in this notebook so it works in a clean kernel with no
    # dependency on fp_neb_generation_and_run.ipynb having run first.
    #
    # Serialization-level comparison for merge-time conflict detection, not
    # a scientific structure matcher: no StructureMatcher remapping, no
    # species reordering. Tolerances are for catching a genuinely different
    # or corrupted structure, not for judging physical equivalence.
    a_sites = structure_dict.get("sites", [])
    b_sites = reference_structure_dict.get("sites", [])
    if len(a_sites) != len(b_sites):
        raise MergeError(f"{context}: atom count mismatch ({len(a_sites)} vs {len(b_sites)})")

    a_species = [s["species"][0]["element"] for s in a_sites]
    b_species = [s["species"][0]["element"] for s in b_sites]
    if a_species != b_species:
        raise MergeError(f"{context}: species/atom-order mismatch ({a_species} vs {b_species})")

    a_lattice = np.asarray(structure_dict["lattice"]["matrix"], dtype=float)
    b_lattice = np.asarray(reference_structure_dict["lattice"]["matrix"], dtype=float)
    if a_lattice.shape != b_lattice.shape or not np.allclose(a_lattice, b_lattice, atol=lattice_atol):
        raise MergeError(f"{context}: lattice matrix mismatch")

    a_frac = np.asarray([s["abc"] for s in a_sites], dtype=float)
    b_frac = np.asarray([s["abc"] for s in b_sites], dtype=float)
    delta = a_frac - b_frac
    delta -= np.round(delta)   # periodic (minimum-image) difference
    if np.any(np.abs(delta) > frac_coord_atol):
        max_delta = float(np.abs(delta).max())
        raise MergeError(
            f"{context}: fractional-coordinate mismatch (max periodic delta="
            f"{max_delta:.6f}, tolerance={frac_coord_atol})"
        )

    if image_index is not None and reference_image_index is not None and image_index != reference_image_index:
        raise MergeError(f"{context}: image_index mismatch ({image_index} vs {reference_image_index})")

    if source_identifiers is not None and reference_identifiers is not None:
        a_pkey = source_identifiers.get("pathway_key")
        b_pkey = reference_identifiers.get("pathway_key")
        if a_pkey is not None and b_pkey is not None and a_pkey != b_pkey:
            raise MergeError(f"{context}: pathway_key identifier mismatch ({a_pkey} vs {b_pkey})")

    return True


def make_dft_static_image_record(image_index, n_images, fp_structure, fp_energy_total_eV,
                                  fp_forces_eV_per_angstrom, dft_energy_total_eV, dft_forces_eV_per_angstrom,
                                  dft_electronic_convergence, calculation_status, calculation_provenance):
    if calculation_status not in CALCULATION_STATUS_VALUES:
        raise ValueError(f"calculation_status must be one of {CALCULATION_STATUS_VALUES}")
    analysis_eligible = bool(calculation_status == "completed" and dft_electronic_convergence is True)
    return {
        "image_index": image_index,
        "endpoint_role": endpoint_role_for(image_index, n_images),
        "fp_structure": fp_structure,
        "fp_energy_total_eV": fp_energy_total_eV,
        "fp_forces_eV_per_angstrom": fp_forces_eV_per_angstrom,
        "dft_energy_total_eV": dft_energy_total_eV,
        "dft_forces_eV_per_angstrom": dft_forces_eV_per_angstrom,
        "dft_electronic_convergence": dft_electronic_convergence,
        "analysis_eligible": analysis_eligible,
        "energy_unit": "eV",
        "force_unit": "eV/angstrom",
        "calculation_status": calculation_status,
        "calculation_provenance": calculation_provenance,
    }


_example_structure = {"lattice": "placeholder", "sites": []}
_example_dft_static = make_dft_static_image_record(
    0, 3, _example_structure, -10.0, [[0.0, 0.0, 0.0]], -10.05, [[0.0, 0.0, 0.0]],
    True, "completed", "dft_static_on_fp_neb.ipynb, example")
print("Example dft_static_on_fp_neb image record:")
print(json.dumps(_example_dft_static, indent=2))


## 3. Configuration

Real Zaratan/VASP paths and SLURM settings. No foundation-potential site-packages paths or model checkpoints are needed; this notebook never loads a foundation-potential model.

In [ ]:
from __future__ import annotations
import copy, os, stat, sys, tempfile, time
from pathlib import Path

import numpy as np
from pymatgen.core import Structure
from pymatgen.io.vasp.sets import MPStaticSet

# The full all-FP standardized results file is not committed to this repo
# (see ../data/README.md for size/checksum/how to obtain it). Download it
# and place it at ../data/ion_migration_neb_results_standardized.json, or
# point this at a candidate written by fp_neb_generation_and_run.ipynb's
# merge step, e.g.:
# FP_RESULTS_FILE = Path("../generation/runs/generated_fp_neb_jobs/merged/ion_migration_neb_fp_results.json")
FP_RESULTS_FILE = Path("../data/ion_migration_neb_results_standardized.json")
# Accepts .json or .json.gz.
DFT_REFERENCE_FILE = Path("../data/ion_migration_neb_reference.json.gz")

# Public safety controls -- see the FP-NEB generation notebook's
# Configuration section for the full rationale. SUBMIT_JOBS and
# PROMOTE_RESULTS gate no in-notebook code path: submission is always an
# external, manual step, and this notebook's merge step always writes a
# new candidate file under runs/.../merged/, never the canonical results.
GENERATE_JOBS            = False  # set True to actually write VASP input directories (Section 6)
WRITE_SUBMISSION_SCRIPTS = False  # set True to write submission_scripts/ (Section 7)
SUBMIT_JOBS              = False  # informational only -- this notebook never submits jobs
PROMOTE_RESULTS          = False  # informational only -- promotion is always a manual, external step

OUTPUT_BASE     = Path("./runs/generated_dft_static_on_fp_neb")
CHECKPOINT_DIR  = Path("./runs/generated_dft_static_on_fp_neb/checkpoints")
MERGED_OUT_DIR  = Path("./runs/generated_dft_static_on_fp_neb/merged")
SUBMISSION_SCRIPTS_DIR = Path("./submission_scripts/dft_static_on_fp_neb")
for d in (OUTPUT_BASE, CHECKPOINT_DIR, MERGED_OUT_DIR, SUBMISSION_SCRIPTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

DFT_STATIC_MERGE_POLICY = "replace_selected_validated"

SLURM_DFT = dict(
    account   = "YOUR_SLURM_ACCOUNT",    # EDIT: your cluster allocation/account
    partition = "YOUR_SLURM_PARTITION",  # EDIT: your cluster's partition/queue name
    ntasks    = 2,
    cpus      = 8,
    mem       = "4G",
    time      = "48:00:00",
)

STATIC_INCAR_SETTINGS = {
    "NSW": 0, "IBRION": -1, "ISMEAR": 0, "SIGMA": 0.05,
    "EDIFF": 1e-6, "LWAVE": False, "LCHARG": False, "LORBIT": 11,
}

VASP_MODULE_LOAD_LINES = "\n".join([
    "module load vasp",
    "export OMPI_MCA_mpi_cuda_support=0",
    "export OMP_NUM_THREADS=$SLURM_CPUS_PER_TASK",
    "export OMP_STACKSIZE=1024M",
    "export OMP_SCHEDULE=static",
    "cd $SLURM_SUBMIT_DIR",
    ". ~/.bashrc",
    "chmod u=rwx /path/to/vasp_std",   # EDIT: your VASP executable path
    "srun /path/to/vasp_std",          # EDIT: your VASP executable path
])

if DFT_REFERENCE_FILE.suffix == ".gz":
    import gzip
    with gzip.open(DFT_REFERENCE_FILE, "rt") as f:
        reference_data = json.load(f)
else:
    with open(DFT_REFERENCE_FILE) as f:
        reference_data = json.load(f)

print(f"FP_RESULTS_FILE exists: {FP_RESULTS_FILE.exists()}")
print(f"DFT_STATIC_MERGE_POLICY: {DFT_STATIC_MERGE_POLICY!r}")


## 4. Load Standardized All-FP Results

Explicit, no fallback: the selected input path, dataset identity, foundation-potential selection, and pathway count are printed before any job is generated. Foundation-potential keys are read directly from `results["models"]`, never hardcoded.

In [ ]:
if not FP_RESULTS_FILE.exists():
    raise FileNotFoundError(
        f"{FP_RESULTS_FILE} does not exist. Download the standardized all-FP results "
        f"file (see ../data/README.md) and place it at "
        f"../data/ion_migration_neb_results_standardized.json, or point FP_RESULTS_FILE "
        f"at a candidate written by fp_neb_generation_and_run.ipynb's merge step; this "
        f"notebook does not fall back to any other results file."
    )

with open(FP_RESULTS_FILE) as f:
    all_fp_results = json.load(f)

fp_keys = list(all_fp_results["models"].keys())

print(f"Input results file : {FP_RESULTS_FILE.resolve()}")
print(f"Dataset name        : {all_fp_results.get('dataset_name')}")
print(f"Schema version       : {all_fp_results.get('schema_version')}")
print(f"FP keys in models     : {fp_keys}")
n_full_fp_neb_pathways = {fp: len(all_fp_results['models'][fp]['full_fp_neb']['pathways']) for fp in fp_keys}
n_unsuccessful = {fp: len(all_fp_results['models'][fp]['full_fp_neb'].get('unsuccessful_pathways', {})) for fp in fp_keys}
for fp in fp_keys:
    print(f"  {fp:22s} full_fp_neb completed={n_full_fp_neb_pathways[fp]:3d}  unsuccessful={n_unsuccessful[fp]:3d}")
print(f"Total completed full_fp_neb pathway records across all FPs: {sum(n_full_fp_neb_pathways.values())}")


## 5. Select Eligible Final FP-NEB Pathways

A pathway is eligible when its `full_fp_neb` record has `neb_status.calculation_status == "completed"` and a complete, correctly indexed `final_fp_neb_images` set, regardless of `neb_converged`: a non-converged full FP-NEB path can still have complete final images suitable for this diagnostic. Failed, missing, interrupted, or not-run paths never reach `pathways` at all (they live in the sibling `unsuccessful_pathways` dict) and are excluded here with an explicit reason.

`SELECTION_MODE`: `"all"` (every eligible path for the selected FPs), `"one"` (single pathway via `SELECTED_PATHWAYS`), `"list"` (explicit list), `"converged"`, `"non_converged"`, or `"both"`.

In [ ]:
SELECTION_MODE = "all"
SELECTED_FP_KEYS = fp_keys
SELECTED_PATHWAYS = []


def eligible_paths_for_fp(fp_key, fp_block, not_run_report):
    eligible = {}
    full_block = fp_block.get("full_fp_neb", {})
    for pathway_key, pdata in full_block.get("pathways", {}).items():
        neb_status = pdata.get("neb_status", {})
        images = pdata.get("final_fp_neb_images", {})
        if not images:
            not_run_report.append({"fp_key": fp_key, "pathway_key": pathway_key,
                                    "reason": "present in pathways but final_fp_neb_images is empty"})
            continue
        expected = set(str(i) for i in range(len(images)))
        if set(images.keys()) != expected:
            not_run_report.append({"fp_key": fp_key, "pathway_key": pathway_key,
                                    "reason": f"structurally incomplete image-index set: {sorted(images.keys())}"})
            continue
        eligible[pathway_key] = (images, neb_status.get("neb_converged"))

    for pathway_key, urec in full_block.get("unsuccessful_pathways", {}).items():
        not_run_report.append({"fp_key": fp_key, "pathway_key": pathway_key,
                                "reason": f"full_fp_neb calculation_status={urec.get('calculation_status')!r}"})
    return eligible


def apply_selection_mode(eligible, mode, selected_pathways):
    if mode in ("all", "both"):
        return eligible
    if mode == "one":
        if len(selected_pathways) != 1:
            raise ValueError('SELECTION_MODE "one" requires exactly one entry in SELECTED_PATHWAYS')
        return {k: v for k, v in eligible.items() if k in set(selected_pathways)}
    if mode == "list":
        return {k: v for k, v in eligible.items() if k in set(selected_pathways)}
    if mode == "converged":
        return {k: v for k, v in eligible.items() if v[1] is True}
    if mode == "non_converged":
        return {k: v for k, v in eligible.items() if v[1] is False}
    raise ValueError(f"Unknown SELECTION_MODE: {mode!r}")


not_run_report = []
target_population = {}
for fp_key in SELECTED_FP_KEYS:
    eligible = eligible_paths_for_fp(fp_key, all_fp_results["models"][fp_key], not_run_report)
    selected = apply_selection_mode(eligible, SELECTION_MODE, SELECTED_PATHWAYS)
    target_population[fp_key] = selected
    n_conv = sum(1 for v in selected.values() if v[1] is True)
    n_nc = sum(1 for v in selected.values() if v[1] is False)
    print(f"  {fp_key:22s} selected={len(selected):3d}  (converged={n_conv}, non_converged={n_nc})")

total_combinations = sum(len(v) for v in target_population.values())
print(f"\nSELECTION_MODE={SELECTION_MODE!r}  total (fp, pathway) combinations selected: {total_combinations}")
print(f"Paths excluded with an explicit reason: {len(not_run_report)}")


## 6. Generate DFT Static Calculations

Real `MPStaticSet` with `STATIC_INCAR_SETTINGS`, real `SLURM_DFT` account and partition, real VASP module-load lines.

A job's electronic-convergence status is determined only by parsing its own real VASP output (Section 8); no historical or inferred warning is attached at generation time.

`MPStaticSet.write_input()` can fail when `PMG_VASP_PSP_DIR` is not configured (`pymatgen.io.vasp.inputs.PmgVaspPspDirError`, a subclass of `ValueError`); only that specific exception is caught and falls back to `POTCAR.spec`. Every other exception is re-raised, not swallowed. A job whose real `POTCAR` could not be written gets `input_generation_status: "not_ready_missing_potcar"` in `job_metadata.json` and is never treated as ready.

Every job directory gets a `job_metadata.json` carrying the unsanitized `pathway_key`, identifiers, and reproducibility metadata (package versions, VASP executable/module, INCAR overrides, POTCAR symbols, source-structure and generated-file SHA-256 hashes, a real generation timestamp). This is the single source of truth both the checkpoint writer (Section 8) and the submission scripts (Section 7) read the true `pathway_key` from; directory names are sanitized for filesystem safety and are never parsed back into identifiers.

In [ ]:
import hashlib
import platform
from datetime import datetime, timezone

import pymatgen
import importlib.metadata
from pymatgen.io.vasp.inputs import PmgVaspPspDirError

VASP_EXECUTABLE_PATH = "/path/to/vasp_std"   # EDIT: your VASP executable path
VASP_MODULE_NAME = "vasp"
REQUIRED_JOB_FILES = ("INCAR", "KPOINTS", "POSCAR", "POTCAR", "slurm.sh", "job_metadata.json")


def make_dft_static_slurm(job_id, job_dir_abs):
    required_check_lines = []
    for fname in REQUIRED_JOB_FILES:
        if fname == "slurm.sh":
            continue
        required_check_lines.append(
            f'if [[ ! -s "{job_dir_abs}/{fname}" ]]; then '
            f'echo "FATAL: required input {fname} is missing or empty, refusing to run VASP" >&2; exit 87; fi'
        )
    lines_ = [
        "#!/bin/bash",
        f"#SBATCH --job-name=dftst_{job_id[:24]}",
        f"#SBATCH -A {SLURM_DFT['account']}",
        f"#SBATCH --ntasks={SLURM_DFT['ntasks']}",
        f"#SBATCH --cpus-per-task={SLURM_DFT['cpus']}",
        f"#SBATCH -p {SLURM_DFT['partition']}",
        f"#SBATCH --mem-per-cpu={SLURM_DFT['mem']}",
        f"#SBATCH -t {SLURM_DFT['time']}",
        "#SBATCH --output=slurm_%j.out",
        "",
        "# Required-input preflight: stop before srun if anything is missing or empty.",
    ] + required_check_lines + [
        "",
        VASP_MODULE_LOAD_LINES,
    ]
    return "\n".join(lines_) + "\n"


def sha256_of_file(path):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return None
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()


def sha256_of_structure(structure_dict):
    canonical = json.dumps(structure_dict, sort_keys=True)
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()


if GENERATE_JOBS:
    n_dirs_written = 0
    n_not_ready_missing_potcar = 0
    generated_job_dirs = []
    for fp_key, selected in target_population.items():
        for pathway_key, (images, _neb_converged) in selected.items():
            safe_key = pathway_key.replace("|", "_p")
            for idx_str, img in images.items():
                job_dir = OUTPUT_BASE / fp_key / safe_key / f"image_{idx_str}"
                job_dir.mkdir(parents=True, exist_ok=True)

                struct = Structure.from_dict(img["structure"])
                vs = MPStaticSet(struct, user_incar_settings=STATIC_INCAR_SETTINGS)

                input_generation_status = "ready"
                generation_error = None
                try:
                    vs.write_input(str(job_dir))
                except PmgVaspPspDirError as e:
                    vs.incar.write_file(str(job_dir / "INCAR"))
                    vs.kpoints.write_file(str(job_dir / "KPOINTS"))
                    vs.poscar.write_file(str(job_dir / "POSCAR"))
                    try:
                        spec = "\n".join(vs.potcar_symbols) + "\n"
                    except Exception:
                        spec = "# determine manually\n"
                    (job_dir / "POTCAR.spec").write_text(spec)
                    input_generation_status = "not_ready_missing_potcar"
                    generation_error = str(e)
                    n_not_ready_missing_potcar += 1
                # Any other exception (malformed structure, unsupported
                # element, disk error, etc.) is not caught here and propagates,
                # so it is never silently mistaken for a missing-POTCAR case.

                try:
                    potcar_symbols = list(vs.potcar_symbols)
                except Exception:
                    potcar_symbols = None
                potcar_hash = sha256_of_file(job_dir / "POTCAR") if input_generation_status == "ready" else None

                job_metadata = {
                    "source_fp_key": fp_key,
                    "pathway_key": pathway_key,
                    "image_index": int(idx_str),
                    "endpoint_role": endpoint_role_for(int(idx_str), len(images)),
                    "source_results_file": str(FP_RESULTS_FILE.resolve()),
                    "source_structure_hash": sha256_of_structure(img["structure"]),
                    "input_generation_status": input_generation_status,
                    "generation_error": generation_error,
                    "python_version": platform.python_version(),
                    "pymatgen_version": importlib.metadata.version("pymatgen"),
                    "vasp_executable_path": VASP_EXECUTABLE_PATH,
                    "vasp_module_name": VASP_MODULE_NAME,
                    "input_set_class": type(vs).__name__,
                    "user_incar_overrides": STATIC_INCAR_SETTINGS,
                    "potcar_symbols": potcar_symbols,
                    "potcar_sha256": potcar_hash,
                    "sha256_incar": sha256_of_file(job_dir / "INCAR"),
                    "sha256_kpoints": sha256_of_file(job_dir / "KPOINTS"),
                    "sha256_poscar": sha256_of_file(job_dir / "POSCAR"),
                    "sha256_potcar": potcar_hash,
                    "generation_timestamp_utc": datetime.now(timezone.utc).isoformat(),
                    "generator": "dft_static_on_fp_neb.ipynb",
                }
                with open(job_dir / "job_metadata.json", "w") as f:
                    json.dump(job_metadata, f, indent=2)

                job_dir_abs = str(job_dir.resolve())
                slurm_path = job_dir / "slurm.sh"
                slurm_path.write_text(make_dft_static_slurm(f"{fp_key}_{safe_key}_img{idx_str}", job_dir_abs))
                slurm_path.chmod(slurm_path.stat().st_mode | stat.S_IXUSR)

                generated_job_dirs.append((fp_key, pathway_key, idx_str, job_dir))
                n_dirs_written += 1

    print(f"Wrote {n_dirs_written} VASP static input directories across "
          f"{sum(len(v) for v in target_population.values())} selected paths.")
    print(f"input_generation_status = 'not_ready_missing_potcar': {n_not_ready_missing_potcar} "
          f"(POTCAR unavailable; POTCAR.spec retained; never submitted as ready)")
else:
    print("GENERATE_JOBS is False -- skipping VASP input-directory generation. "
          "Set GENERATE_JOBS = True in Configuration to write real job directories.")
    n_dirs_written = 0
    generated_job_dirs = []


## 7. Submission Scripts

Bash submission interfaces under `submission_scripts/dft_static_on_fp_neb/`. FP selection identifies the source foundation potential whose `full_fp_neb` pathways supply the structures; no foundation-potential model is loaded to submit or run these jobs.

The checkpoint key is never reconstructed from the sanitized directory name: the decision helper reads the real, unsanitized `pathway_key` from each job's own `job_metadata.json` (written in Section 6) and builds the key with the same `checkpoint_record_key(fp_key, pathway_key, image_index)` format the parser uses (Section 8), so the two can never silently disagree.

The existence of `vasprun.xml` is not treated as proof of success: each decision consults the parsed checkpoint. By default, only images with a checkpointed `analysis_eligible: true` are skipped; corrupt or electronically non-converged VASP outputs, and images never yet parsed, are resubmitted, not silently skipped. `--retry-non-converged` narrows submission to only checkpointed `completed` records with `dft_electronic_convergence: false`. `--force` resubmits everything. A job whose `job_metadata.json` reports `input_generation_status != "ready"`, or whose required input files are missing or empty, is never submitted, `--force` included.

In [ ]:
def write_lines(path, py_lines):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(py_lines) + "\n")
    path.chmod(path.stat().st_mode | stat.S_IXUSR)
    return path


_DECISION_HELPER = (
    "import json, os, sys\n"
    "\n"
    "def checkpoint_record_key(fp_key, pathway_key, image_index):\n"
    "    return f'{fp_key}|{pathway_key}|{image_index}'\n"
    "\n"
    "ckpt_path, job_dir, retry_nc = sys.argv[1], sys.argv[2], sys.argv[3] == '1'\n"
    "\n"
    "meta_path = os.path.join(job_dir, 'job_metadata.json')\n"
    "if not os.path.exists(meta_path):\n"
    "    print('NOT_READY missing job_metadata.json')\n"
    "    sys.exit(0)\n"
    "with open(meta_path) as f:\n"
    "    meta = json.load(f)\n"
    "\n"
    "if meta.get('input_generation_status') != 'ready':\n"
    "    print('NOT_READY input_generation_status=' + str(meta.get('input_generation_status')))\n"
    "    sys.exit(0)\n"
    "\n"
    "required = ('INCAR', 'KPOINTS', 'POSCAR', 'POTCAR', 'slurm.sh', 'job_metadata.json')\n"
    "missing = [r for r in required if not os.path.exists(os.path.join(job_dir, r))\n"
    "           or os.path.getsize(os.path.join(job_dir, r)) == 0]\n"
    "if missing:\n"
    "    print('NOT_READY missing_or_empty=' + ','.join(missing))\n"
    "    sys.exit(0)\n"
    "\n"
    "ckey = checkpoint_record_key(meta['source_fp_key'], meta['pathway_key'], meta['image_index'])\n"
    "\n"
    "records = {}\n"
    "if os.path.exists(ckpt_path):\n"
    "    with open(ckpt_path) as f:\n"
    "        records = json.load(f).get('records', {})\n"
    "rec = records.get(ckey)\n"
    "if retry_nc:\n"
    "    ok = bool(rec and rec.get('calculation_status') == 'completed' and rec.get('dft_electronic_convergence') is False)\n"
    "    print('SUBMIT' if ok else 'SKIP')\n"
    "else:\n"
    "    eligible = bool(rec and rec.get('analysis_eligible') is True)\n"
    "    print('SKIP' if eligible else 'SUBMIT')\n"
)


def common_sh_lines_dft_static(job_root, slurm_basename, checkpoint_path):
    lines_ = [
        "#!/bin/bash",
        "set -euo pipefail",
        "",
        'PROTOCOL="dft_static_on_fp_neb"',
        f'JOB_ROOT="{job_root}"',
        f'SLURM_BASENAME="{slurm_basename}"',
        f'CHECKPOINT_PATH="{checkpoint_path}"',
        'FP_KEYS_FILE="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)/fp_keys.txt"',
        'DECISION_HELPER="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)/_decision.py"',
        "",
        "DRY_RUN=0", "FORCE=0", "RETRY_NON_CONVERGED=0", 'LABEL=""', "REQUESTED_KEYS=()",
        "",
        "usage() {",
        '  echo "Usage: $0 [--dry-run] [--force] [--retry-non-converged] [--label LABEL] SOURCE_FP_KEY [SOURCE_FP_KEY ...]" >&2',
        "  exit 1",
        "}",
        "",
        "while [[ $# -gt 0 ]]; do",
        '  case "$1" in',
        "    --dry-run) DRY_RUN=1; shift ;;",
        "    --force) FORCE=1; shift ;;",
        "    --retry-non-converged) RETRY_NON_CONVERGED=1; shift ;;",
        "    --label) LABEL=\"$2\"; shift 2 ;;",
        "    -h|--help) usage ;;",
        "    *) REQUESTED_KEYS+=(\"$1\"); shift ;;",
        "  esac",
        "done",
        "",
        "if [[ ${#REQUESTED_KEYS[@]} -eq 0 ]]; then usage; fi",
        "",
        "VALID_KEYS=()",
        "while IFS= read -r line; do",
        '  [[ -n "$line" ]] && VALID_KEYS+=("$line")',
        'done < "$FP_KEYS_FILE"',
        "",
        "key_in_list() {",
        '  local needle="$1"; shift',
        "  local candidate",
        '  for candidate in "$@"; do',
        '    [[ "$candidate" == "$needle" ]] && return 0',
        "  done",
        "  return 1",
        "}",
        "",
        'for k in "${REQUESTED_KEYS[@]}"; do',
        '  if ! key_in_list "$k" "${VALID_KEYS[@]}"; then',
        '    echo "ERROR: unknown source FP key \'$k\'. Valid keys: ${VALID_KEYS[*]}" >&2',
        "    exit 1",
        "  fi",
        "  n_occurrences=0",
        '  for k2 in "${REQUESTED_KEYS[@]}"; do',
        '    [[ "$k2" == "$k" ]] && n_occurrences=$((n_occurrences+1))',
        "  done",
        '  if [[ "$n_occurrences" -gt 1 ]]; then',
        '    echo "ERROR: duplicate source FP key \'$k\' requested" >&2',
        "    exit 1",
        "  fi",
        "done",
        "",
        'if [[ -z "$LABEL" ]]; then LABEL="$(date +%Y%m%d_%H%M%S)"; fi',
        'LOG_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)/logs/${LABEL}"',
        'mkdir -p "$LOG_DIR"',
        'LOG_FILE="${LOG_DIR}/submission.log"',
        "",
        'echo "Run label: $LABEL"',
        'echo "Protocol: $PROTOCOL"',
        'echo "Requested source FP keys: ${REQUESTED_KEYS[*]}"',
        'echo "Dry run: $DRY_RUN   Force: $FORCE   Retry non-converged: $RETRY_NON_CONVERGED"',
        "",
        "n_submitted=0",
        "n_skipped=0",
        "n_not_ready=0",
        "",
        'for fp_key in "${REQUESTED_KEYS[@]}"; do',
        '  fp_root="${JOB_ROOT}/${fp_key}"',
        '  if [[ ! -d "$fp_root" ]]; then',
        '    echo "WARNING: no job directory for source fp=$fp_key under $fp_root" >&2',
        "    continue",
        "  fi",
        '  while IFS= read -r -d "" slurm_path; do',
        '    job_dir="$(dirname "$slurm_path")"',
        '    image_id="$(basename "$job_dir")"',
        '    pathway_id="$(basename "$(dirname "$job_dir")")"',
        '    decision_full="$(python3 "$DECISION_HELPER" "$CHECKPOINT_PATH" "$job_dir" "$RETRY_NON_CONVERGED")"',
        '    decision="${decision_full%% *}"',
        '    if [[ "$decision" == "NOT_READY" ]]; then',
        '      echo "NOT READY, not submitted: source_fp=$fp_key pathway=$pathway_id image=$image_id ($decision_full)" >&2',
        "      n_not_ready=$((n_not_ready+1))",
        "      continue",
        "    fi",
        '    if [[ "$FORCE" -eq 0 ]] && [[ "$decision" == "SKIP" ]]; then',
        "      n_skipped=$((n_skipped+1))",
        "      continue",
        "    fi",
        '    if [[ "$DRY_RUN" -eq 1 ]]; then',
        '      echo "[DRY RUN] would submit: protocol=$PROTOCOL source_fp=$fp_key pathway=$pathway_id image=$image_id dir=$job_dir"',
        "      n_submitted=$((n_submitted+1))",
        "      continue",
        "    fi",
        '    job_id="$(cd "$job_dir" && sbatch "$SLURM_BASENAME" | awk \'{print $NF}\')"',
        '    ts="$(date -u +%Y-%m-%dT%H:%M:%SZ)"',
        '    echo "${ts} protocol=${PROTOCOL} source_fp=${fp_key} pathway=${pathway_id} image=${image_id} dir=${job_dir} slurm_job_id=${job_id}" >> "$LOG_FILE"',
        '    echo "Submitted: source_fp=$fp_key pathway=$pathway_id image=$image_id slurm_job_id=$job_id"',
        "    n_submitted=$((n_submitted+1))",
        f'  done < <(find "$fp_root" -name "$SLURM_BASENAME" -print0 | sort -z)',
        "done",
        "",
        'echo "Done. Submitted=$n_submitted Skipped(checkpointed analysis_eligible)=$n_skipped NotReady(missing inputs)=$n_not_ready"',
        'echo "Log: $LOG_FILE"',
    ]
    return lines_


_dft_checkpoint_path = CHECKPOINT_DIR / "dft_static_on_fp_neb_checkpoint.json"

if WRITE_SUBMISSION_SCRIPTS:
    SUBMISSION_SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)
    (SUBMISSION_SCRIPTS_DIR / "fp_keys.txt").write_text("\n".join(fp_keys) + "\n")
    (SUBMISSION_SCRIPTS_DIR / "_decision.py").write_text(_DECISION_HELPER)

    write_lines(SUBMISSION_SCRIPTS_DIR / "_common.sh",
                common_sh_lines_dft_static(str(OUTPUT_BASE.resolve()), "slurm.sh", str(_dft_checkpoint_path.resolve())))

    write_lines(SUBMISSION_SCRIPTS_DIR / "submit_one_source_fp.sh", [
        "#!/bin/bash", "set -euo pipefail",
        'DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"',
        "if [[ $# -lt 1 ]]; then",
        '  echo "Usage: $0 SOURCE_FP_KEY [--dry-run] [--force] [--retry-non-converged] [--label LABEL]" >&2',
        "  exit 1", "fi",
        'FP_KEY="$1"; shift',
        'exec "$DIR/_common.sh" "$@" "$FP_KEY"',
    ])
    write_lines(SUBMISSION_SCRIPTS_DIR / "submit_selected_source_fps.sh", [
        "#!/bin/bash", "set -euo pipefail",
        'DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"',
        "if [[ $# -lt 1 ]]; then",
        '  echo "Usage: $0 SOURCE_FP_KEY [SOURCE_FP_KEY ...] [--dry-run] [--force] [--retry-non-converged] [--label LABEL]" >&2',
        "  exit 1", "fi",
        'exec "$DIR/_common.sh" "$@"',
    ])
    write_lines(SUBMISSION_SCRIPTS_DIR / "submit_all_source_fps.sh", [
        "#!/bin/bash", "set -euo pipefail",
        'DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"',
        "ALL_KEYS=()",
        "while IFS= read -r line; do",
        '  [[ -n "$line" ]] && ALL_KEYS+=("$line")',
        'done < "$DIR/fp_keys.txt"',
        'exec "$DIR/_common.sh" "$@" "${ALL_KEYS[@]}"',
    ])

    print(f"Wrote submission scripts: {SUBMISSION_SCRIPTS_DIR}/"
          "{_common.sh,_decision.py,submit_one_source_fp.sh,submit_selected_source_fps.sh,submit_all_source_fps.sh,fp_keys.txt}")
    print()
    print("Example: submission_scripts/dft_static_on_fp_neb/submit_one_source_fp.sh MACE-MP0_medium --dry-run")
    print("Example: submission_scripts/dft_static_on_fp_neb/submit_selected_source_fps.sh MACE-MP0_medium CHGNET --dry-run")
    print("Example: submission_scripts/dft_static_on_fp_neb/submit_all_source_fps.sh --dry-run")
else:
    print("WRITE_SUBMISSION_SCRIPTS is False -- skipping submission_scripts/ generation. "
          "Set WRITE_SUBMISSION_SCRIPTS = True in Configuration to write them.")


## 8. Parse and Checkpoint DFT Results

Real parser against `pymatgen.io.vasp.Vasprun`. Before a result can be `analysis_eligible: true`, it must pass every one of: `vasprun.xml` parses; `dft_energy_total_eV` is finite; the force array is numeric with shape `(n_atoms, 3)`; the parsed atom count equals the source FP image's atom count; the parsed final structure corresponds to the generated static-input structure within serialization tolerance (a static, `NSW=0` calculation should not move the ions; if it did, or species/order differ, that is treated as a parsing failure, not a valid but non-converged result); `dft_electronic_convergence is True`; no required identifier or unit is missing. `Vasprun.converged_electronic` is the authoritative electronic-convergence flag.

Execution and electronic convergence are kept separate: a VASP process can finish (`calculation_status: "completed"`) with `dft_electronic_convergence: False`; such a record stays fully traceable but `analysis_eligible` is `False` and it must not enter valid DFT force-error statistics. A corrupt or structurally inconsistent `vasprun.xml` is recorded as `failed`, never `completed`. Missing output never fabricates energy or forces. This cell runs the same way whether zero, some, or all generated jobs have real output: it contains no assertion requiring a particular count of completed or failed records.

The checkpoint key is built with `checkpoint_record_key(fp_key, pathway_key, image_index)` using values read from each job's own `job_metadata.json`, the same canonical function and the same source the submission decision helper (Section 7) uses -- the two can never silently disagree.

In [ ]:
from pymatgen.io.vasp import Vasprun
from pymatgen.io.ase import AseAtomsAdaptor


def checkpoint_record_key(fp_key, pathway_key, image_index):
    return f"{fp_key}|{pathway_key}|{image_index}"


def parse_vasprun_result(vasprun_path, source_structure_dict, context):
    vr = Vasprun(str(vasprun_path), parse_potcar_file=False)

    energy = float(vr.final_energy)
    if not np.isfinite(energy):
        raise ValueError(f"{context}: parsed dft_energy_total_eV is not finite ({energy})")

    last_step = vr.ionic_steps[-1] if vr.ionic_steps else None
    forces = last_step["forces"] if last_step is not None else vr.as_dict()["output"].get("forces")
    forces_arr = np.asarray(forces, dtype=float)

    source_structure = Structure.from_dict(source_structure_dict)
    n_atoms_source = len(source_structure)
    if forces_arr.shape != (n_atoms_source, 3):
        raise ValueError(f"{context}: force array shape {forces_arr.shape} != expected ({n_atoms_source}, 3)")
    if not np.all(np.isfinite(forces_arr)):
        raise ValueError(f"{context}: force array contains non-finite values")

    final_structure = vr.final_structure
    if len(final_structure) != n_atoms_source:
        raise ValueError(f"{context}: parsed atom count {len(final_structure)} != source atom count {n_atoms_source}")

    # A static (NSW=0) calculation must not have moved the ions; species,
    # ordering, lattice, and periodic fractional coordinates are checked
    # against the generated input, not merely the atom count.
    validate_structure_atoms(final_structure.as_dict(), source_structure_dict,
                              f"{context}: parsed final structure vs generated input")

    return {
        "dft_energy_total_eV": energy,
        "dft_forces_eV_per_angstrom": forces_arr.tolist(),
        "dft_electronic_convergence": bool(vr.converged_electronic),
    }


def dft_status_for_job(job_dir, source_structure_dict, context):
    job_dir = Path(job_dir)
    if not job_dir.exists():
        return "not_run", None
    vasprun_path = job_dir / "vasprun.xml"
    if not vasprun_path.exists():
        return "missing", None
    try:
        return "completed", parse_vasprun_result(vasprun_path, source_structure_dict, context)
    except Exception as e:
        # Any parsing/validation failure (corrupt XML, non-finite energy,
        # wrong force shape, atom-count mismatch, unexpected ionic motion,
        # species/order mismatch) is recorded as failed, never completed.
        return "failed", {"error": f"{type(e).__name__}: {e}"}


def load_checkpoint(checkpoint_path):
    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.exists():
        return {"records": {}}
    with open(checkpoint_path) as f:
        return json.load(f)


def atomic_write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp_path = tempfile.mkstemp(dir=str(path.parent), prefix=path.name + ".tmp")
    try:
        with os.fdopen(fd, "w") as f:
            json.dump(data, f, indent=2)
            f.flush()
            os.fsync(f.fileno())
        os.replace(tmp_path, path)
    except Exception:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
        raise


dft_checkpoint = load_checkpoint(_dft_checkpoint_path)

parsed_records = {}
status_counts = {"completed": 0, "failed": 0, "missing": 0, "not_run": 0}
for fp_key, pathway_key, idx_str, job_dir in generated_job_dirs:
    with open(job_dir / "job_metadata.json") as f:
        job_meta = json.load(f)
    assert job_meta["source_fp_key"] == fp_key and job_meta["pathway_key"] == pathway_key, (
        f"job_metadata.json identifiers do not match the in-memory loop values for {job_dir}"
    )

    images = target_population[fp_key][pathway_key][0]
    source_structure_dict = images[idx_str]["structure"]
    context = f"{fp_key} {pathway_key} image {idx_str}"

    if job_meta.get("input_generation_status") != "ready":
        status, parsed = "missing", {"error": f"input_generation_status={job_meta.get('input_generation_status')!r}"}
    else:
        status, parsed = dft_status_for_job(job_dir, source_structure_dict, context)
    parsed_records[(fp_key, pathway_key, idx_str)] = {"status": status, "parsed": parsed}
    status_counts[status] += 1

    dft_electronic_convergence = (parsed or {}).get("dft_electronic_convergence")
    ckey = checkpoint_record_key(job_meta["source_fp_key"], job_meta["pathway_key"], job_meta["image_index"])
    dft_checkpoint["records"][ckey] = {
        "fp_key": job_meta["source_fp_key"], "pathway_key": job_meta["pathway_key"],
        "image_index": job_meta["image_index"],
        "calculation_status": status,
        "dft_energy_total_eV": (parsed or {}).get("dft_energy_total_eV"),
        "dft_forces_eV_per_angstrom": (parsed or {}).get("dft_forces_eV_per_angstrom"),
        "dft_electronic_convergence": dft_electronic_convergence,
        "analysis_eligible": bool(status == "completed" and dft_electronic_convergence is True),
        "error": (parsed or {}).get("error"),
        "job_metadata_reference": str((job_dir / "job_metadata.json").resolve()),
    }

atomic_write_json(_dft_checkpoint_path, dft_checkpoint)
print("Parsed status across generated jobs:", status_counts)

# General validation, holds both before and after real Zaratan results exist.
for (fp_key, pathway_key, idx_str), rec in parsed_records.items():
    if rec["status"] == "completed":
        assert rec["parsed"].get("dft_energy_total_eV") is not None, f"{fp_key} {pathway_key} img{idx_str}: completed but no parsed energy"
        assert rec["parsed"].get("dft_forces_eV_per_angstrom") is not None, f"{fp_key} {pathway_key} img{idx_str}: completed but no parsed forces"
        assert rec["parsed"].get("dft_electronic_convergence") is not None, f"{fp_key} {pathway_key} img{idx_str}: completed but no electronic-convergence flag"
    if rec["status"] == "missing":
        assert rec["parsed"] is None or "error" in rec["parsed"], f"{fp_key} {pathway_key} img{idx_str}: missing status must not carry fabricated energy/forces"
        assert (rec["parsed"] or {}).get("dft_energy_total_eV") is None, f"{fp_key} {pathway_key} img{idx_str}: missing status must not carry a fabricated energy"
print("General validation passed: completed records carry parsed energy/forces/convergence; missing records fabricate nothing.")

for ckey, rec in dft_checkpoint["records"].items():
    if rec["dft_electronic_convergence"] is False:
        assert rec["calculation_status"] == "completed", f"{ckey}: non-converged record must still be calculation_status=completed"
        assert rec["analysis_eligible"] is False, f"{ckey}: electronically non-converged record must be analysis_eligible=false"
print("Confirmed: electronically non-converged records remain completed/traceable but analysis_eligible=false.")


## 9. Validate Structure and Image Correspondence

Confirms the structure a VASP job was generated for is, atom-for-atom, the same structure recorded in the source `full_fp_neb` image before any DFT result is merged back in, using the canonical `validate_structure_atoms` from Section 2. Also confirms every job directory has the required files for a runnable Zaratan VASP job (`INCAR`, `KPOINTS`, `POSCAR`, `POTCAR`, `slurm.sh`, `job_metadata.json`, all nonempty) unless `input_generation_status == "not_ready_missing_potcar"`, in which case the missing `POTCAR` is expected and reported, not treated as a mismatch.

In [ ]:
def validate_structure_correspondence(fp_key, pathway_key, idx_str, img, job_dir):
    poscar_path = Path(job_dir) / "POSCAR"
    if not poscar_path.exists() or poscar_path.stat().st_size == 0:
        return False, "POSCAR missing or empty"
    written = Structure.from_file(str(poscar_path))
    try:
        validate_structure_atoms(written.as_dict(), img["structure"],
                                  f"{fp_key} {pathway_key} image {idx_str}: generated POSCAR vs source")
    except MergeError as e:
        return False, str(e)
    return True, "OK"


n_checked, n_mismatch = 0, 0
n_not_ready, n_missing_required = 0, 0
for fp_key, pathway_key, idx_str, job_dir in generated_job_dirs:
    images = target_population[fp_key][pathway_key][0]
    ok, detail = validate_structure_correspondence(fp_key, pathway_key, idx_str, images[idx_str], job_dir)
    n_checked += 1
    if not ok:
        n_mismatch += 1
        print(f"  MISMATCH {fp_key} {pathway_key} img{idx_str}: {detail}")

    with open(job_dir / "job_metadata.json") as f:
        job_meta = json.load(f)
    if job_meta["input_generation_status"] != "ready":
        n_not_ready += 1
        continue
    for fname in REQUIRED_JOB_FILES:
        fpath = job_dir / fname
        if not fpath.exists() or fpath.stat().st_size == 0:
            n_missing_required += 1
            print(f"  NOT READY {fp_key} {pathway_key} img{idx_str}: {fname} missing or empty despite input_generation_status='ready'")

print(f"Structure correspondence validated for {n_checked} images, {n_mismatch} mismatches.")
print(f"Jobs with input_generation_status != 'ready': {n_not_ready}")
print(f"Jobs marked 'ready' but missing a required file: {n_missing_required}")
assert n_mismatch == 0
assert n_missing_required == 0, "a job marked 'ready' is missing a required file; input_generation_status is inconsistent with disk state"


## 10. Merge Into the All-Protocol Results File

`DFT_STATIC_MERGE_POLICY = 'replace_selected_validated'`: writes to a new output file and never modifies the input results file in place; replaces only selected image records for which a new result was successfully parsed and validated; requires exact FP, pathway, image, structure, atom-count, atom-order, force-shape, and image-index/endpoint-role correspondence against any existing record before replacing it; preserves every unselected existing record unchanged; preserves an old valid record when a new attempt is missing, failed, or unparsable, recording the action as `retained` plus a separate `failed_new_attempt` entry; records whether each image was `added`, `replaced`, `retained`, or `rejected`; rejects (does not overwrite, logs loudly) a conflicting source structure or identifiers. `full_fp_neb` and `fp_static_on_dft_neb` are always copied through unchanged.

Every attempt that does not become an active `pathways` record is additionally preserved in a sibling `unsuccessful_image_attempts` branch (FP key, pathway key, image index, status, error, and provenance) inside the same output file, outside the branch the analysis loader reads -- an in-memory merge-action list alone is not the preservation mechanism.

In [ ]:
# MergeError and validate_structure_atoms are defined in Section 2; reused
# here unchanged, not redefined, so there is exactly one validator.
assert DFT_STATIC_MERGE_POLICY == "replace_selected_validated", (
    f"Unknown DFT_STATIC_MERGE_POLICY: {DFT_STATIC_MERGE_POLICY!r}"
)

updated_results = copy.deepcopy(all_fp_results)
merge_actions = []   # (fp_key, pathway_key, idx_str, action, detail)

for fp_key in fp_keys:
    updated_results["models"][fp_key]["dft_static_on_fp_neb"].setdefault("unsuccessful_image_attempts", {})

for fp_key, pathway_key, idx_str, job_dir in generated_job_dirs:
    images = target_population[fp_key][pathway_key][0]
    n_images = len(images)
    fp_structure = images[idx_str]["structure"]
    fp_energy = images[idx_str]["fp_energy_total_eV"]
    fp_forces = images[idx_str]["fp_forces_eV_per_angstrom"]
    image_index = int(idx_str)

    dest_model = updated_results["models"][fp_key]["dft_static_on_fp_neb"]["pathways"]
    unsuccessful_dest = updated_results["models"][fp_key]["dft_static_on_fp_neb"]["unsuccessful_image_attempts"]
    existing_pathway = dest_model.get(pathway_key)
    existing_image = (existing_pathway or {}).get("images", {}).get(idx_str)
    attempt_key = f"{fp_key}|{pathway_key}|{idx_str}"

    if existing_image is not None:
        try:
            validate_structure_atoms(
                fp_structure, existing_image["fp_structure"], f"{fp_key} {pathway_key} image {idx_str}",
                image_index=image_index, reference_image_index=existing_image.get("image_index"),
            )
            existing_forces = np.asarray(existing_image.get("fp_forces_eV_per_angstrom", []))
            new_forces = np.asarray(fp_forces)
            if existing_forces.shape != new_forces.shape:
                raise MergeError(f"{fp_key} {pathway_key} image {idx_str}: force-array shape conflict "
                                  f"({new_forces.shape} vs existing {existing_forces.shape})")
        except MergeError as e:
            merge_actions.append((fp_key, pathway_key, idx_str, "rejected", f"conflicting source structure: {e}"))
            unsuccessful_dest[attempt_key] = {
                "fp_key": fp_key, "pathway_key": pathway_key, "image_index": image_index,
                "status": "rejected", "error": str(e),
                "calculation_provenance": "dft_static_on_fp_neb.ipynb Section 10 merge, structure/identifier conflict",
            }
            continue

    record = parsed_records[(fp_key, pathway_key, idx_str)]
    if record["status"] != "completed":
        error_detail = (record.get("parsed") or {}).get("error")
        if existing_image is not None:
            merge_actions.append((fp_key, pathway_key, idx_str, "retained",
                                   f"new attempt status={record['status']!r}, old valid record preserved"))
            merge_actions.append((fp_key, pathway_key, idx_str, "failed_new_attempt", error_detail or record["status"]))
        else:
            merge_actions.append((fp_key, pathway_key, idx_str, "skipped_no_data",
                                   f"new attempt status={record['status']!r}, nothing to write"))
        unsuccessful_dest[attempt_key] = {
            "fp_key": fp_key, "pathway_key": pathway_key, "image_index": image_index,
            "status": record["status"], "error": error_detail,
            "calculation_provenance": "dft_static_on_fp_neb.ipynb Section 8 parse",
        }
        continue

    parsed = record["parsed"]
    new_image = make_dft_static_image_record(
        image_index, n_images, fp_structure, fp_energy, fp_forces,
        parsed["dft_energy_total_eV"], parsed["dft_forces_eV_per_angstrom"],
        parsed["dft_electronic_convergence"], "completed",
        calculation_provenance="dft_static_on_fp_neb.ipynb, parsed from real vasprun.xml",
    )

    if pathway_key not in dest_model:
        source_identifiers = all_fp_results["models"][fp_key]["full_fp_neb"]["pathways"][pathway_key]["identifiers"]
        dest_model[pathway_key] = {"identifiers": make_identifiers(source_identifiers), "images": {}}
    dest_model[pathway_key]["images"][idx_str] = new_image
    unsuccessful_dest.pop(attempt_key, None)   # a later successful attempt supersedes any earlier failed one
    merge_actions.append((fp_key, pathway_key, idx_str, "replaced" if existing_image is not None else "added", "ok"))

action_counts = {}
for *_r, action, _detail in merge_actions:
    action_counts[action] = action_counts.get(action, 0) + 1
print("Merge actions:", action_counts)
print("unsuccessful_image_attempts recorded per FP:",
      {fp: len(updated_results["models"][fp]["dft_static_on_fp_neb"]["unsuccessful_image_attempts"]) for fp in fp_keys})

for fp_key in fp_keys:
    assert updated_results["models"][fp_key]["full_fp_neb"] == all_fp_results["models"][fp_key]["full_fp_neb"]
    assert updated_results["models"][fp_key]["fp_static_on_dft_neb"] == all_fp_results["models"][fp_key]["fp_static_on_dft_neb"]
print("Confirmed: full_fp_neb and fp_static_on_dft_neb copied through unchanged.")

merged_out_path = MERGED_OUT_DIR / "ion_migration_neb_fp_results.json"
assert merged_out_path.resolve() != FP_RESULTS_FILE.resolve(), "merge output must not be the same path as the input file"
with open(merged_out_path, "w") as f:
    json.dump(updated_results, f, indent=2)
print(f"Wrote {merged_out_path} (new file; {FP_RESULTS_FILE} was not modified)")

# Confirm the input file on disk is byte-identical to before the merge.
with open(FP_RESULTS_FILE) as f:
    _post_merge_input = json.load(f)
assert _post_merge_input == all_fp_results, "input results file was modified by this notebook"
print(f"Confirmed {FP_RESULTS_FILE} was not modified.")


## 11. Validate Generated Jobs and Results

Generated-file and schema validation, performed before submission; real VASP/SLURM outcomes are recorded only after execution on Zaratan. Includes a synthetic exercise of the merge policy (added/replaced/retained/rejected) against a pre-populated results file, since 0 real VASP results exist on this machine.

In [ ]:
checks = []
def check(name, ok, detail=""):
    checks.append({"check": name, "ok": bool(ok), "detail": str(detail)})

check("FP keys read directly from results['models'], not hardcoded", fp_keys == list(all_fp_results["models"].keys()))
check("endpoint_role vocabulary is initial/intermediate/final",
      {endpoint_role_for(0, 3), endpoint_role_for(1, 3), endpoint_role_for(2, 3)} == {"initial", "intermediate", "final"})

_dft_schema_fields = {"image_index", "endpoint_role", "fp_structure", "fp_energy_total_eV",
                       "fp_forces_eV_per_angstrom", "dft_energy_total_eV", "dft_forces_eV_per_angstrom",
                       "dft_electronic_convergence", "analysis_eligible", "energy_unit", "force_unit",
                       "calculation_status", "calculation_provenance"}
check("dft_static image record matches documented schema exactly", set(_example_dft_static.keys()) == _dft_schema_fields)
check("'partial' is accepted by CALCULATION_STATUS_VALUES", "partial" in CALCULATION_STATUS_VALUES)

if generated_job_dirs:
    n_missing_input_files = sum(1 for _, _, _, d in generated_job_dirs if not (d / "INCAR").exists() and not (d / "POTCAR.spec").exists())
    check("every generated job dir has INCAR or POTCAR.spec", n_missing_input_files == 0, n_missing_input_files)

import subprocess
n_bad_sh = 0
for sh in SUBMISSION_SCRIPTS_DIR.rglob("*.sh"):
    r = subprocess.run(["bash", "-n", str(sh)], capture_output=True, text=True)
    if r.returncode != 0:
        n_bad_sh += 1
    check(f"submission script passes bash -n: {sh.relative_to(SUBMISSION_SCRIPTS_DIR)}", r.returncode == 0, r.stderr[:200])

check("merged output file is valid JSON, one complete all-protocol file", isinstance(json.load(open(merged_out_path)), dict))

import sys as _sys
_sys.path.insert(0, "../scripts")
import neb_analysis as na
_dft_ref, _fp_res = na.load_neb_datasets(DFT_REFERENCE_FILE, merged_out_path)
check("merged results file loads via neb_analysis.load_neb_datasets, no reshaping", isinstance(_fp_res, dict))

# ── Synthetic exercise: successfully parsed, electronically non-converged,
#    failed, and missing results, plus resume-behavior and merge-policy
#    tests, since 0 real VASP results exist on this machine.
_sample_pkey = reference_data["common_pathway_keys"][0]
_sample_pdata = reference_data["pathways"][_sample_pkey]
_sample_images = sorted(_sample_pdata["dft_neb_reference"]["images"], key=lambda im: im["image_index"])
_synthetic_fp = fp_keys[0]

_synthetic_all_results = copy.deepcopy(all_fp_results)
_synthetic_full_images = {
    str(im["image_index"]): {
        "image_index": im["image_index"], "endpoint_role": endpoint_role_for(im["image_index"], len(_sample_images)),
        "structure": im["structure"], "fp_energy_total_eV": im["energy_total_eV"],
        "fp_forces_eV_per_angstrom": im["forces_eV_per_angstrom"], "energy_unit": "eV", "force_unit": "eV/angstrom",
        "calculation_provenance": "synthetic test",
    } for im in _sample_images
}
_synthetic_all_results["models"][_synthetic_fp]["full_fp_neb"]["pathways"][_sample_pkey] = {
    "identifiers": _sample_pdata["identifiers"], "final_fp_neb_images": _synthetic_full_images,
    "neb_status": {"calculation_status": "completed", "neb_converged": False},
}

_probe = []
_eligible = eligible_paths_for_fp(_synthetic_fp, _synthetic_all_results["models"][_synthetic_fp], _probe)
check("synthetic: non-converged completed path is eligible", _sample_pkey in _eligible)

# Four synthetic parse outcomes over the same pathway's four images.
_synthetic_parsed = {
    "0": {"status": "completed", "parsed": {"dft_energy_total_eV": -10.0, "dft_forces_eV_per_angstrom": [[0, 0, 0]],
                                             "dft_electronic_convergence": True}},
    "1": {"status": "completed", "parsed": {"dft_energy_total_eV": -9.5, "dft_forces_eV_per_angstrom": [[0, 0, 0]],
                                             "dft_electronic_convergence": False}},
    "2": {"status": "failed", "parsed": {"error": "corrupt vasprun.xml"}},
    "3": {"status": "missing", "parsed": None},
}
_rows = []
for idx_str, rec in _synthetic_parsed.items():
    if rec["status"] == "completed":
        img = make_dft_static_image_record(
            int(idx_str), len(_sample_images), _synthetic_full_images[idx_str]["structure"],
            _synthetic_full_images[idx_str]["fp_energy_total_eV"], _synthetic_full_images[idx_str]["fp_forces_eV_per_angstrom"],
            rec["parsed"]["dft_energy_total_eV"], rec["parsed"]["dft_forces_eV_per_angstrom"],
            rec["parsed"]["dft_electronic_convergence"], "completed", "synthetic test")
        _rows.append((idx_str, img))
check("synthetic: successfully-parsed image is analysis_eligible=True", dict(_rows)["0"]["analysis_eligible"] is True)
check("synthetic: electronically non-converged image is analysis_eligible=False, still traceable",
      dict(_rows)["1"]["analysis_eligible"] is False and dict(_rows)["1"]["dft_electronic_convergence"] is False)
check("synthetic: failed parse produces no fabricated energy/forces (status tracked separately)",
      _synthetic_parsed["2"]["parsed"].get("dft_energy_total_eV") is None)
check("synthetic: missing result is not fabricated", _synthetic_parsed["3"]["parsed"] is None)

# Merge-policy exercise: pre-populate a results file with one existing
# dft_static_on_fp_neb image, then merge a new synthetic parse over it.
_pre_existing = copy.deepcopy(_synthetic_all_results)
_pre_existing_img = dict(_rows)["0"]
_pre_existing["models"][_synthetic_fp]["dft_static_on_fp_neb"]["pathways"][_sample_pkey] = {
    "identifiers": _sample_pdata["identifiers"], "images": {"0": _pre_existing_img},
}
_dest = _pre_existing["models"][_synthetic_fp]["dft_static_on_fp_neb"]["pathways"][_sample_pkey]["images"]
check("merge policy: pre-existing record present before re-merge", "0" in _dest)
# Simulate a "retained" outcome (new attempt failed, old valid record kept):
_new_attempt_status = "failed"
_retained = "0" in _dest and _new_attempt_status != "completed"
check("merge policy: old valid record retained when new attempt fails", _retained)
# Simulate a "rejected" outcome (conflicting source structure). Uses
# except MergeError specifically -- a NameError from a missing function
# must never be interpreted as a successfully-raised conflict, which is
# exactly the class of bug this notebook previously had (Section 1 of the
# accompanying bug-fix report): validate_structure_atoms was called but
# never defined in this notebook, so every call here would have raised
# NameError, and a blanket "except Exception" would have wrongly reported
# every one of these checks as passing.
try:
    validate_structure_atoms({"sites": [{"species": [{"element": "Xx"}]}]}, _pre_existing_img["fp_structure"], "synthetic conflict test")
    _rejected_raises = False
except MergeError:
    _rejected_raises = True
except NameError:
    _rejected_raises = False   # function missing: must count as a real failure, not success
check("merge policy: conflicting source structure raises MergeError specifically, not silently overwritten", _rejected_raises)

# Meta-test: prove the test methodology itself distinguishes a genuine
# MergeError from an unrelated exception (a NameError must not be
# swallowed and reinterpreted as "conflict correctly detected").
def _meta_check_merge_error_specificity():
    try:
        raise NameError("intentionally simulated missing-function error")
    except MergeError:
        return False   # wrong: a NameError must never satisfy an "except MergeError" branch
    except NameError:
        return True     # correct: NameError is distinguishable from MergeError
check("test methodology: NameError is never mistaken for a raised MergeError", _meta_check_merge_error_specificity())

check("SELECTION_MODE is one of the documented values", SELECTION_MODE in ("all", "one", "list", "converged", "non_converged", "both"))

# ── Item 1 acceptance tests: identical / coordinate-mismatched / different-
#    lattice / different-ordering structures, using real pymatgen structures.
from pymatgen.core import Lattice as _Lattice
_s_ref = Structure(_Lattice.cubic(4.0), ["Li", "Cl"], [[0, 0, 0], [0.5, 0.5, 0.5]]).as_dict()
_s_identical = Structure(_Lattice.cubic(4.0), ["Li", "Cl"], [[0, 0, 0], [0.5, 0.5, 0.5]]).as_dict()
_s_moved = Structure(_Lattice.cubic(4.0), ["Li", "Cl"], [[0, 0, 0], [0.4, 0.5, 0.5]]).as_dict()
_s_diff_lattice = Structure(_Lattice.cubic(4.5), ["Li", "Cl"], [[0, 0, 0], [0.5, 0.5, 0.5]]).as_dict()
_s_diff_order = Structure(_Lattice.cubic(4.0), ["Cl", "Li"], [[0.5, 0.5, 0.5], [0, 0, 0]]).as_dict()

check("validator: identical structures pass", validate_structure_atoms(_s_ref, _s_identical, "test") is True)
try:
    validate_structure_atoms(_s_ref, _s_moved, "test"); _ok_moved = False
except MergeError:
    _ok_moved = True
check("validator: same-species coordinate-mismatched structure raises MergeError", _ok_moved)
try:
    validate_structure_atoms(_s_ref, _s_diff_lattice, "test"); _ok_lat = False
except MergeError:
    _ok_lat = True
check("validator: different lattice raises MergeError", _ok_lat)
try:
    validate_structure_atoms(_s_ref, _s_diff_order, "test"); _ok_order = False
except MergeError:
    _ok_order = True
check("validator: different atom ordering raises MergeError", _ok_order)

# ── Item 3: generated-input validation against manuscript-consistent
#    MPStaticSet settings. Uses a representative real written INCAR/KPOINTS
#    from this run's own generated_job_dirs when any eligible pathway
#    exists; otherwise (0 real eligible full_fp_neb pathways on this
#    machine, the honest common case) builds one synthetic job directory
#    with the exact same generation code as Section 6, so this validation
#    always genuinely runs rather than being silently skipped.
import shutil as _shutil
from pymatgen.io.vasp.inputs import Incar, Kpoints
_incar_dirs = [d for _, _, _, d in generated_job_dirs if (d / "INCAR").exists() and (d / "INCAR").stat().st_size > 0]
if not _incar_dirs:
    # No real eligible full_fp_neb pathway exists on this machine (the
    # honest common case before real Zaratan results exist). Build one
    # complete synthetic job directory with the exact same generation code
    # as Section 6 (INCAR/KPOINTS/POSCAR/POTCAR.spec, job_metadata.json,
    # slurm.sh) so every downstream check in this section genuinely runs
    # rather than being silently skipped.
    _synthetic_job_dir = OUTPUT_BASE / "_synthetic_input_validation_probe"
    if _synthetic_job_dir.exists():
        _shutil.rmtree(_synthetic_job_dir)
    _synthetic_job_dir.mkdir(parents=True)
    _synthetic_struct = Structure.from_dict(_s_ref)
    _synthetic_vs = MPStaticSet(_synthetic_struct, user_incar_settings=STATIC_INCAR_SETTINGS)
    _synthetic_status = "ready"
    try:
        _synthetic_vs.write_input(str(_synthetic_job_dir))
    except PmgVaspPspDirError:
        _synthetic_vs.incar.write_file(str(_synthetic_job_dir / "INCAR"))
        _synthetic_vs.kpoints.write_file(str(_synthetic_job_dir / "KPOINTS"))
        _synthetic_vs.poscar.write_file(str(_synthetic_job_dir / "POSCAR"))
        try:
            _spec = "\n".join(_synthetic_vs.potcar_symbols) + "\n"
        except Exception:
            _spec = "# determine manually\n"
        (_synthetic_job_dir / "POTCAR.spec").write_text(_spec)
        _synthetic_status = "not_ready_missing_potcar"
    _synthetic_meta = {
        "source_fp_key": "SYNTHETIC_TEST_FP", "pathway_key": "999999|1", "image_index": 0,
        "endpoint_role": "initial", "source_results_file": "synthetic",
        "source_structure_hash": sha256_of_structure(_s_ref),
        "input_generation_status": _synthetic_status, "generation_error": None,
        "python_version": platform.python_version(), "pymatgen_version": importlib.metadata.version("pymatgen"),
        "vasp_executable_path": VASP_EXECUTABLE_PATH, "vasp_module_name": VASP_MODULE_NAME,
        "input_set_class": type(_synthetic_vs).__name__, "user_incar_overrides": STATIC_INCAR_SETTINGS,
        "potcar_symbols": None, "potcar_sha256": None,
        "sha256_incar": sha256_of_file(_synthetic_job_dir / "INCAR"),
        "sha256_kpoints": sha256_of_file(_synthetic_job_dir / "KPOINTS"),
        "sha256_poscar": sha256_of_file(_synthetic_job_dir / "POSCAR"),
        "sha256_potcar": None,
        "generation_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "generator": "dft_static_on_fp_neb.ipynb (Section 11 synthetic probe)",
    }
    with open(_synthetic_job_dir / "job_metadata.json", "w") as f:
        json.dump(_synthetic_meta, f, indent=2)
    (_synthetic_job_dir / "slurm.sh").write_text(make_dft_static_slurm("synthetic_probe", str(_synthetic_job_dir.resolve())))
    (_synthetic_job_dir / "slurm.sh").chmod((_synthetic_job_dir / "slurm.sh").stat().st_mode | stat.S_IXUSR)
    _incar_dirs = [_synthetic_job_dir]
    print("No real eligible full_fp_neb pathway on this machine; using one synthetic "
          "MPStaticSet-generated job directory to validate manuscript-consistent settings.")

_sample_job_dir = _incar_dirs[0]
_incar = Incar.from_file(str(_sample_job_dir / "INCAR"))
check("generated INCAR: NSW = 0", _incar.get("NSW") == 0, _incar.get("NSW"))
check("generated INCAR: IBRION = -1", _incar.get("IBRION") == -1, _incar.get("IBRION"))
check("generated INCAR: ISMEAR = 0", _incar.get("ISMEAR") == 0, _incar.get("ISMEAR"))
check("generated INCAR: SIGMA = 0.05", abs(_incar.get("SIGMA", -1) - 0.05) < 1e-9, _incar.get("SIGMA"))
check("generated INCAR: EDIFF = 1e-6", abs(_incar.get("EDIFF", -1) - 1e-6) < 1e-12, _incar.get("EDIFF"))
check("generated INCAR: ENCUT = 520", abs(_incar.get("ENCUT", -1) - 520) < 1e-6, _incar.get("ENCUT"))
check("generated INCAR: spin polarization enabled (ISPIN = 2)", _incar.get("ISPIN") == 2, _incar.get("ISPIN"))
if (_sample_job_dir / "KPOINTS").exists():
    _kpts = Kpoints.from_file(str(_sample_job_dir / "KPOINTS"))
    check("generated KPOINTS: Gamma-centered", str(_kpts.style).lower().startswith("gamma") or "gamma" in str(_kpts.style).lower(),
          str(_kpts.style))
check("no ionic relaxation requested (NSW=0 and IBRION=-1 together)",
      _incar.get("NSW") == 0 and _incar.get("IBRION") == -1)

# ── Item 4: fail-safe VASP input generation and required-file preflight.
check("PmgVaspPspDirError is the specific exception caught for missing POTCAR",
      issubclass(PmgVaspPspDirError, ValueError))
_not_ready_dirs = [d for fp, pw, idx, d in generated_job_dirs
                    if json.load(open(d / "job_metadata.json"))["input_generation_status"] != "ready"]
if not _not_ready_dirs and not (_sample_job_dir / "POTCAR").exists():
    # The synthetic probe dir also demonstrates the missing-POTCAR case on
    # this machine (no PMG_VASP_PSP_DIR configured here).
    _not_ready_dirs = [_sample_job_dir]
if _not_ready_dirs:
    _nr = _not_ready_dirs[0]
    check("a not-ready job retains POTCAR.spec but has no real POTCAR",
          (_nr / "POTCAR.spec").exists() and not (_nr / "POTCAR").exists())
    _nr_meta = json.load(open(_nr / "job_metadata.json"))
    check("not-ready job_metadata.json reports not_ready_missing_potcar",
          _nr_meta["input_generation_status"] == "not_ready_missing_potcar")
    _nr_decision = subprocess.run(
        ["python3", str(SUBMISSION_SCRIPTS_DIR / "_decision.py"), str(_dft_checkpoint_path), str(_nr), "0"],
        capture_output=True, text=True).stdout.strip()
    check("submission decision helper refuses to submit a not-ready (missing POTCAR) job",
          _nr_decision.startswith("NOT_READY"), _nr_decision)
# "Ready" job test: no PMG_VASP_PSP_DIR is configured on this machine, so
# no locally generated job (real or synthetic) ever has a genuine POTCAR.
# To test the ready-path decision/preflight logic without claiming a real
# POTCAR was produced, build a copy with a placeholder (clearly fake, not
# licensed) nonempty POTCAR file and job_metadata.json's
# input_generation_status forced to "ready" -- this tests the logic given
# a job_metadata.json that reports ready, not a real POTCAR generation.
_ready_probe_dir = OUTPUT_BASE / "_ready_job_probe"
if _ready_probe_dir.exists():
    _shutil.rmtree(_ready_probe_dir)
_shutil.copytree(_sample_job_dir, _ready_probe_dir)
(_ready_probe_dir / "POTCAR").write_text("# placeholder for local testing only, not real POTCAR content\n")
_ready_probe_meta = json.load(open(_ready_probe_dir / "job_metadata.json"))
_ready_probe_meta["input_generation_status"] = "ready"
with open(_ready_probe_dir / "job_metadata.json", "w") as f:
    json.dump(_ready_probe_meta, f, indent=2)

check("a ready job has input_generation_status == 'ready'", _ready_probe_meta["input_generation_status"] == "ready")
_ready_decision = subprocess.run(
    ["python3", str(SUBMISSION_SCRIPTS_DIR / "_decision.py"), str(_dft_checkpoint_path), str(_ready_probe_dir), "0"],
    capture_output=True, text=True).stdout.strip()
check("submission decision helper allows a ready, unparsed job to be submitted",
      _ready_decision == "SUBMIT", _ready_decision)

# slurm.sh preflight: doctor a further copy to be missing POSCAR and
# confirm the generated slurm.sh script itself stops before srun (exit
# 87), never silently proceeding.
_tmp_probe_dir = OUTPUT_BASE / "_slurm_preflight_probe"
if _tmp_probe_dir.exists():
    _shutil.rmtree(_tmp_probe_dir)
_shutil.copytree(_ready_probe_dir, _tmp_probe_dir)
(_tmp_probe_dir / "POSCAR").unlink()
_preflight_result = subprocess.run(["bash", "-n", str(_tmp_probe_dir / "slurm.sh")], capture_output=True, text=True)
check("generated slurm.sh (with POSCAR removed) still passes bash -n", _preflight_result.returncode == 0)
_slurm_text = (_tmp_probe_dir / "slurm.sh").read_text()
check("generated slurm.sh contains a required-file preflight before srun",
      "FATAL" in _slurm_text and "srun" in _slurm_text and _slurm_text.index("FATAL") < _slurm_text.rindex("srun"))
_shutil.rmtree(_tmp_probe_dir)

# ── Item 2/11: checkpoint key built from job_metadata.json exactly matches
#    between the Python parser and the bash decision helper, for a real
#    FPBench-style pathway key containing "|". Falls back to the synthetic
#    probe (pathway_key "999999|1", still containing "|") when no real
#    eligible pathway exists on this machine.
if generated_job_dirs:
    _fp0, _pw0, _idx0, _d0 = generated_job_dirs[0]
else:
    _fp0, _pw0, _idx0, _d0 = "SYNTHETIC_TEST_FP", "999999|1", "0", _synthetic_job_dir
check("test pathway key contains '|' (real FPBench-style identifier)", "|" in _pw0, _pw0)
_py_ckey = checkpoint_record_key(_fp0, _pw0, int(_idx0))
_meta0 = json.load(open(_d0 / "job_metadata.json"))
_py_ckey_from_meta = checkpoint_record_key(_meta0["source_fp_key"], _meta0["pathway_key"], _meta0["image_index"])
check("checkpoint key from job_metadata.json matches the in-memory key", _py_ckey == _py_ckey_from_meta,
      f"{_py_ckey} vs {_py_ckey_from_meta}")
check("job_metadata.json pathway_key is unsanitized (contains '|', not '_p')",
      "|" in _meta0["pathway_key"] and "_p" not in _meta0["pathway_key"].split("|")[-1])

_ckey_functional = subprocess.run(
    ["python3", str(SUBMISSION_SCRIPTS_DIR / "_decision.py"), str(_dft_checkpoint_path), str(_d0), "0"],
    capture_output=True, text=True).stdout.strip()
check("submission decision helper resolves a checkpoint key for a pathway key containing '|' without error",
      len(_ckey_functional) > 0, _ckey_functional)

# ── Item 6: corrupt vasprun.xml and incomplete force array are recorded as
#    failed parsing, never completed.
_tmp_corrupt = OUTPUT_BASE / "_corrupt_vasprun_probe"
_tmp_corrupt.mkdir(exist_ok=True)
(_tmp_corrupt / "vasprun.xml").write_text("<not-valid-xml-at-all")
_corrupt_status, _corrupt_parsed = dft_status_for_job(_tmp_corrupt, _s_ref, "corrupt test")
check("corrupt vasprun.xml is recorded as failed, not completed", _corrupt_status == "failed", _corrupt_status)
import shutil as _shutil2
_shutil2.rmtree(_tmp_corrupt)

# Clean up local-only test probe directories (not part of any real
# generated job population).
for _probe in (OUTPUT_BASE / "_synthetic_input_validation_probe", OUTPUT_BASE / "_ready_job_probe"):
    if _probe.exists():
        _shutil.rmtree(_probe)

report_ok_before = list(fp_keys)

import pandas as pd
report_df = pd.DataFrame(checks)
print(report_df.to_string(index=False))
n_fail = (~report_df["ok"]).sum()
print(f"\n{len(report_df)} checks run, {n_fail} failed.")
assert n_fail == 0
